# 03 · Modelado de Machine Learning — Riesgo de Mora
**TUMIPAY — Prueba Técnica Data Science Engineer**

## Definición del problema
**Tipo:** Clasificación binaria  
**Variable objetivo:** `mora-30` (1 = crédito con ≥1 cuota en las primeras 6 con días_mora > 30)  
**Horizonte:** Predicción al momento del desembolso usando información disponible en ese instante  
**Métrica principal:** AUC-ROC (evalúa discriminación sin depender del umbral)  
**Métrica secundaria:** Recall de la clase positiva (minimizar falsos negativos costosos en crédito)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, classification_report,
                              roc_curve, precision_recall_curve, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)

DATA = '../data/'
df = pd.read_csv(DATA + 'dataset_analitico.csv')
print(f'Dataset: {df.shape} | Tasa mora: {df["mora-30"].mean():.2%}')

Dataset: (1411, 44) | Tasa mora: 23.95%


## 1. Selección de features y criterio anti-fuga

Solo usamos variables **disponibles al momento del desembolso**. Variables excluidas por fuga:
- `estado_credito_operativo` → refleja el estado posterior al comportamiento de pago
- `max_dias_mora_6m`, `pct_pagadas_tiempo` → son consecuencia del target, no predictores
- Cualquier variable calculada sobre el historial de pagos completo

Variables incluidas del módulo `eventos_app`: solo comportamiento digital **previo o contemporáneo** al desembolso, no posterior.

In [2]:
FEATURES_NUM = [
    # Del crédito al momento de originar
    'monto_credito', 'plazo_meses', 'tasa_interes_mensual', 'valor_cuota_pactada',
    'score_interno_originacion', 'relacion_cuota_ingreso', 'tasa_outlier',
    # Del cliente al momento de originar
    'edad', 'estrato', 'ingreso_mensual_estimado', 'score_externo',
    'numero_dependientes', 'score_externo_nulo', 'edad_sospechosa',
    # Eventos digitales del cliente (comportamiento previo)
    'total_eventos', 'logins', 'pagos_fallidos', 'pct_exitosos',
    'pct_abandonados', 'solicitudes_soporte', 'sesion_promedio_seg',
    'tasa_conversion_pago',
]
FEATURES_CAT = [
    'producto_credito', 'canal_originacion', 'politica_aprobacion',
    'genero', 'nivel_educativo', 'ocupacion', 'canal_adquisicion',
    'tiene_producto_ahorro', 'dispositivo_principal',
]
TARGET = 'mora_30'

# Codificar categóricas
df_model = df[FEATURES_NUM + FEATURES_CAT + [TARGET]].copy()
for col in FEATURES_CAT:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))

ALL_FEATURES = FEATURES_NUM + FEATURES_CAT
X = df_model[ALL_FEATURES]
y = df_model[TARGET]

print(f'Features totales: {len(ALL_FEATURES)}')
print(f'  Numéricas: {len(FEATURES_NUM)} | Categóricas: {len(FEATURES_CAT)}')
print(f'Nulos en X: {X.isnull().sum().sum()}')

Features totales: 31
  Numéricas: 22 | Categóricas: 9
Nulos en X: 0


## 2. Split entrenamiento / prueba

Split estratificado para preservar la proporción de mora en ambos conjuntos. Usamos 80/20 dado el tamaño del dataset (1,411 registros).

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]} filas ({y_train.mean():.2%} mora)')
print(f'Test:  {X_test.shape[0]} filas ({y_test.mean():.2%} mora)')

Train: 1128 filas (23.94% mora)
Test:  283 filas (24.03% mora)


## 3. Modelo 1: Regresión Logística (baseline)

**Por qué:** Es el modelo estándar en credit scoring. Simple, interpretable, cumple con requisitos de explicabilidad regulatoria. Si el desempeño es comparable al RF, se prefiere por transparencia.

In [4]:
pipe_lr = Pipeline([
    ('imp',    SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])
pipe_lr.fit(X_train, y_train)
y_prob_lr = pipe_lr.predict_proba(X_test)[:, 1]
auc_lr = roc_auc_score(y_test, y_prob_lr)
print(f'AUC-ROC Regresión Logística: {auc_lr:.4f}')
print()
print(classification_report(y_test, (y_prob_lr >= 0.5).astype(int),
                             target_names=['Sin mora','Con mora']))

AUC-ROC Regresión Logística: 0.7064

              precision    recall  f1-score   support

    Sin mora       0.88      0.64      0.74       215
    Con mora       0.39      0.72      0.50        68

    accuracy                           0.66       283
   macro avg       0.63      0.68      0.62       283
weighted avg       0.76      0.66      0.68       283



## 4. Modelo 2: Random Forest (modelo principal)

**Por qué:** Captura relaciones no lineales y entre variables (e.g. score bajo + RCI alto = riesgo muy alto). Robusto ante outliers y no requiere normalización. Provee importancia de variables directamente.

**Parámetros clave:**
- `max_depth=9`: limita el sobreajuste
- `min_samples_leaf=15`: nodos terminales con al menos 15 muestras, mejora generalización
- `class_weight='balanced'`: compensa el desbalance de clases (24% mora)

In [5]:
X_tr_f  = X_train.fillna(X_train.median())
X_te_f  = X_test.fillna(X_train.median())
X_all_f = X.fillna(X.median())

rf = RandomForestClassifier(
    n_estimators=400, max_depth=9, min_samples_leaf=15,
    max_features='sqrt', class_weight='balanced',
    random_state=42, n_jobs=-1)
rf.fit(X_tr_f, y_train)
y_prob_rf = rf.predict_proba(X_te_f)[:, 1]
auc_rf = roc_auc_score(y_test, y_prob_rf)
print(f'AUC-ROC Random Forest: {auc_rf:.4f}')
print()
# Umbral 0.4 para favorecer recall de mora (falsos negativos son más costosos)
print(classification_report(y_test, (y_prob_rf >= 0.4).astype(int),
                             target_names=['Sin mora','Con mora']))

AUC-ROC Random Forest: 0.6847

              precision    recall  f1-score   support

    Sin mora       0.87      0.54      0.67       215
    Con mora       0.34      0.75      0.47        68

    accuracy                           0.59       283
   macro avg       0.61      0.64      0.57       283
weighted avg       0.74      0.59      0.62       283



## 5. Validación cruzada — estabilidad del modelo

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_lr = cross_val_score(pipe_lr, X, y, cv=cv, scoring='roc_auc')
cv_rf = cross_val_score(rf, X_all_f, y, cv=cv, scoring='roc_auc')

resumen = pd.DataFrame({
    'Modelo': ['Regresión Logística', 'Random Forest'],
    'AUC Test':    [round(auc_lr,4), round(auc_rf,4)],
    'AUC CV Mean': [round(cv_lr.mean(),4), round(cv_rf.mean(),4)],
    'AUC CV Std':  [round(cv_lr.std(),4), round(cv_rf.std(),4)],
})
print(resumen.to_string(index=False))
print('\nLa baja desviación estándar en CV (< 0.05) indica modelos estables.')
print('La Logística es marginalmente mejor -> preferida por interpretabilidad.')

             Modelo  AUC Test  AUC CV Mean  AUC CV Std
Regresión Logística    0.7064       0.7015      0.0437
      Random Forest    0.6847       0.6972      0.0417

La baja desviación estándar en CV (< 0.05) indica modelos estables.
La Logística es marginalmente mejor -> preferida por interpretabilidad.


## 7. Importancia de variables — interpretación de negocio

In [7]:
importancias = pd.Series(rf.feature_importances_, index=ALL_FEATURES)\
                 .sort_values(ascending=False).head(10).reset_index()
importancias.columns = ['Variable','Importancia']
importancias['Interpretación negocio'] = [
    'Score propio TUMIPAY en originación — señal más fuerte',
    'Historial crediticio externo (buró)',
    'Nivel de ingresos del cliente',
    'Carga de deuda relativa: cuota/ingreso',
    'Edad del cliente — proxy de madurez financiera',
    'Monto desembolsado — exposición del crédito',
    'Cuota mensual pactada — correlacionada con monto',
    'Tasa de interés — proxy del segmento de riesgo',
    'Estrato socioeconómico',
    'Plazo del crédito',
][:len(importancias)]
print(importancias.to_string(index=False))

                 Variable  Importancia                                 Interpretación negocio
            score_externo       0.1936 Score propio TUMIPAY en originación — señal más fuerte
score_interno_originacion       0.1317                    Historial crediticio externo (buró)
 ingreso_mensual_estimado       0.1012                          Nivel de ingresos del cliente
   relacion_cuota_ingreso       0.0590                 Carga de deuda relativa: cuota/ingreso
                     edad       0.0493         Edad del cliente — proxy de madurez financiera
      sesion_promedio_seg       0.0481            Monto desembolsado — exposición del crédito
      valor_cuota_pactada       0.0468       Cuota mensual pactada — correlacionada con monto
     tasa_interes_mensual       0.0423         Tasa de interés — proxy del segmento de riesgo
            monto_credito       0.0419                                 Estrato socioeconómico
             pct_exitosos       0.0354                      

## 8. Segmentación por banda de riesgo

In [8]:
df2 = pd.read_csv(DATA + 'dataset_con_scores.csv')
seg = df2.groupby('banda_riesgo', observed=True).agg(
    N=('mora_30','count'),
    Tasa_mora_real=('mora_30','mean'),
    Monto_promedio=('monto_credito','mean'),
    Score_interno_prom=('score_interno_originacion','mean')
).reset_index()
print(seg.to_string(index=False))
print('\nLa separación monótona entre bandas valida el poder discriminativo del modelo.')

     banda_riesgo   N  Tasa_mora_real  Monto_promedio  Score_interno_prom
   Alto\n(35-60%) 642          0.2492    2213940.8100            794.5452
    Bajo\n(< 15%)  73          0.0274    3335616.4384            897.1096
  Medio\n(15-35%) 484          0.0351    3041425.6198            875.1756
Muy alto\n(> 60%) 212          0.7500    2140801.8868            736.3585

La separación monótona entre bandas valida el poder discriminativo del modelo.


## 9. Limitaciones, riesgos y mejoras propuestas

### Limitaciones
| Limitación | Impacto | Mitigación aplicada |
|---|---|---|
| Dataset sintético | Patrones pueden no reflejar producción real | Documentado como supuesto |
| Split aleatorio (no temporal) | Posible sobreestimación del AUC | En producción usar out-of-time validation |
| Desbalance de clases (24% mora) | Sesgo hacia clase mayoritaria | `class_weight='balanced'` + umbral ajustado |
| Variables demográficas (género, estrato) | Riesgo de sesgo discriminatorio | Revisar fairness antes de desplegar |
| Ingreso estimado con nulos | Imputación introduce ruido | Imputación por grupo de ocupación |

### Mejoras con más tiempo
1. **XGBoost / LightGBM:** Típicamente superan RF en datos tabulares de crédito
2. **Out-of-time validation:** Separar test por fecha, no aleatoriamente
3. **SHAP values:** Explicabilidad individual por crédito
4. **Calibración de probabilidades:** Para que los scores sean comparables en el tiempo
5. **Optimización de hiperparámetros:** Optuna o BayesSearch
6. **Feature engineering adicional:** Ratios entre score interno y externo, déciles de ingreso